# DL vertex finding network training

This notebook is designed to take the input and truth images generated by the <code>make_images.ipynb</code> notebook and train networks for vertex finding. This notebook generates models for each of the U, V and W views for each of the required passes.
    
Most of the cells below will not need any editing, but towards the bottom of the notebook you will find some additional markdown that describes what you may need to edit (essentially just some file locations).

In [1]:
# Automatically reload external libraries that change
%reload_ext autoreload
%autoreload 2

# If a matplotlib plot command is issued, display the results in the notebook
%matplotlib inline

In [2]:
# imaging.py

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from tqdm.notebook import tqdm


def imagify(input, pred, truth, n=3, randomize=True, null_code=0):
    """Process input, prediction and mask data ready for display
    
    Args:
        inputs: Input tensor from a batch
        predictions: Predictions tensor from a batch
        truth: Truth tensor from a batch
        n: The number of images to extract from the batch (default: 3)
        randomize: Choose random images from the batch if True, choose the first n otherwise (default: True)
        null_code: The null mask code (default: 0)
            
    Returns:
        A tuple (if n == 1) or zip of the processed images ready for display.
    """
    # Select the images to process
    choices = np.random.choice(np.array(range(inputs.shape[0])), size=n) if randomize else np.array(range(n))
    input_imgs = input[choices,0,...]
    truth_imgs = truth[choices,...]

    input_imgs = input_imgs.detach().cpu()
    truth_imgs = truth_imgs.detach().cpu()
    pred_imgs = pred[choices,...].detach().cpu()

    # Remove non-hit regions
    mask = truth_imgs == null_code
    pred_imgs = np.argmax(pred_imgs, axis=1)
    pred_imgs = np.ma.array(pred_imgs, mask = mask).filled(0)

    return zip(input_imgs, truth_imgs, pred_imgs) if n > 1 else (input_imgs, truth_imgs, pred_imgs)


def show_batch(epoch, batch, input, pred, truth, null_code=0, n=3, randomize=True):
    """Display the images for a given epoch and batch. Each row is a triplet of input, prediction and mask.

    Args:
        epoch: The current training epoch
        batch: The current training batch
        input: Input tensor from a batch
        pred: Predictions tensor from a batch
        truth: Truth tensor from a batch
        n: The number of images to extract from the batch (default: 3)
        randomize: Choose random images from the batch if True, choose the first n otherwise (default: True)
        null_code: The null mask code (default: 0).
    """
    global vertex_pass, view
    ax = None
    rows, cols, size = 1, 2, 9
    cmap = "magma" #ListedColormap(['black', 'red'])
    bounds = np.linspace(0, 19, 19)
    norm = BoundaryNorm(boundaries=bounds, ncolors=19)
    xtr = dict(cmap=cmap, norm=norm)
    #norm = BoundaryNorm([0., 0.05, 1.], cmap.N)
    #cmap = ListedColormap(['black', 'red', 'yellow'])
    #norm = BoundaryNorm([0., 0.5, 1.5, 2.], cmap.N)
    #xtr = dict(cmap=cmap, norm=norm, alpha=0.7)

    images = imagify(input, pred, truth, n, randomize, null_code)

    for i, imgs in enumerate(images):
        raw, cls, net = imgs
        pair = (cls, net)
        fig, axs = plt.subplots(1, cols, figsize=(cols * size, size))
        for img, ax in zip(pair, axs):
            #ax.imshow(raw, cmap="gist_gray")
            ax.imshow(img, **xtr)
            #ax.imshow(img, cmap="magma")
            ax.axis('off')
        plt.tight_layout()
        save_figure(plt, f"outputs/images/pass{vertex_pass}/{view}/output_{epoch}_{batch}_{i}")
        plt.close(fig)


def save_figure(fig, name):
    """Output a matplotlib figure PNG, PDF and EPS formats.

    Args:
        fig (Figure): The matplotlib figure to save.
        name (str): The output filename excluding extension.
    """
    fig.savefig(name + ".png", facecolor='w')
    fig.savefig(name + ".pdf")
    #fig.savefig(name + ".eps")


def get_supported_formats():
    """Retrieve the supported image formats.

    Returns:
        A dictionary containing strings of file format descriptions keyed by extension.
    """
    return plt.gcf().canvas.get_supported_filetypes()

In [3]:
# analysis.py
from functools import partial

def flatten_model(module):
    children = list(module.children())
    if len(children) == 0:
        return [module]
    else:
        flat_model = []
        for child in children:
            flat_model += flatten_model(child)
        return flat_model


class Hook:
    def __init__(self, id, module, func):
        self.id = id
        self.name = module.__class__.__name__
        self.hook = module.register_forward_hook(partial(func, self))
    
    def remove(self):
        self.hook.remove()
    
    def __del__(self):
        self.remove()


def append_stats(hook, module, input, output):
    if not module.training:
        return
    if not hasattr(hook, 'stats'):
        hook.stats = ([],[],[])
    means, stds, hists = hook.stats
    means.append(output.data.mean())
    stds.append(output.data.std())
    hists.append(output.data.histc(40, -5, 5))

In [4]:
# model.py

import torch.nn as nn
import torch


def maxpool():
    """Return a max pooling layer.
    
        The maxpooling layer has a kernel size of 2, a stride of 2 and no padding.

        Returns:
            The max pooling layer
    """
    return nn.MaxPool2d(kernel_size = 2, stride = 2, padding = 0)


def dropout(prob):
    """Return a dropout layer.

        Args:
            prob: The probability that drop out will be applied.

        Returns:
            The dropout layer
    """
    return nn.Dropout(prob)


def reinit_layer(layer, leak = 0.0, use_kaiming_normal=True):
    """Reinitialises convolutional layer weights.
    
        The default Kaiming initialisation in PyTorch is not optimal, this method
        reinitialises the layers using better parameters

        Args:
            seq_block: The layer to be reinitialised.
            leak: The leakiness of ReLU (default: 0.0)
            use_kaiming_normal: Use Kaiming normal if True, Kaiming uniform otherwise (default: True)
    """
    if isinstance(layer, nn.Conv2d) or isinstance(layer, nn.ConvTranspose2d):
        if use_kaiming_normal:
            nn.init.kaiming_normal_(layer.weight, a = leak)
        else:
            nn.init.kaiming_uniform_(layer.weight, a = leak)
            layer.bias.data.zero_()


class ConvBlock(nn.Module):
    """A convolution block
    """
    
    # Sigmoid activation suitable for binary cross-entropy
    def __init__(self, c_in, c_out, k_size = 3, k_pad = 1):
        """Constructor.

            Args:
                c_in: The number of input channels
                c_out: The number of output channels
                k_size: The size of the convolution filter
                k_pad: The amount of padding around the images
        """
        super(ConvBlock, self).__init__()
        self.conv1 = nn.Conv2d(c_in, c_out, kernel_size = k_size, padding = k_pad, stride = 1)
        self.norm1 = nn.GroupNorm(8, c_out)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(c_out, c_out, kernel_size = k_size, padding = k_pad, stride = 1)
        self.norm2 = nn.GroupNorm(8, c_out)
        self.identity = nn.Conv2d(c_in, c_out, kernel_size = 1, padding = 0, stride = 1)
        reinit_layer(self.conv1)
        reinit_layer(self.conv2)

    def forward(self, x):
        """Forward pass.
        
            Args:
                x: The input to the layer
                
            Returns:
                The output from the layer
        """
        identity = self.identity(x)
        x = self.conv1(x)
        x = self.norm1(x)
        x = self.relu(x)
        x = self.conv2(x)
        x = self.norm2(x)
        return self.relu(x + identity)


class TransposeConvBlock(nn.Module):
    """A tranpose convolution block
    """
    
    def __init__(self, c_in, c_out, k_size = 3, k_pad = 1):
        """Constructor.

            Args:
                c_in: The number of input channels
                c_out: The number of output channels
                k_size: The size of the convolution filter
                k_pad: The amount of padding around the images
        """
        super(TransposeConvBlock, self).__init__()
        self.block = nn.Sequential(
            nn.ConvTranspose2d(c_in, c_out, kernel_size = k_size, padding = k_pad, output_padding = 1, stride = 2),
            nn.GroupNorm(8, c_out),
            nn.ReLU(inplace=True))
        reinit_layer(self.block[0])

    def forward(self, x):
        """Forward pass.
        
            Args:
                x: The input to the layer
                
            Returns:
                The output from the layer
        """
        return self.block(x)

class Sigmoid(nn.Module):
    """A sigmoid activation function that supports categorical cross-entropy
    """
    
    def __init__(self, out_range = None):
        """Constructor.

            Args:
                out_range: A tuple covering the minimum and maximum values to map to
        """
        super(Sigmoid, self).__init__()
        if out_range is not None:
            self.low, self.high = out_range
            self.range = self.high - self.low
        else:
            self.low = None
            self.high = None
            self.range = None
    
    def forward(self, x):
        """Applies the sigmoid function.
        
            Rescales to the specified range if provided during construction
        
            Args:
                x: The input to the layer
                
            Returns:
                The (potentially scaled) sigmoid of the input
        """
        if self.low is not None:
            return torch.sigmoid(x) * (self.range) + self.low
        else:
            return torch.sigmoid(x)

class UNet(nn.Module):
    """A U-Net for semantic segmentation.
    """
    
    def __init__(self, in_dim, n_classes, depth = 4, n_filters = 16, drop_prob = 0.1, y_range = None):
        """Constructor.

            Args:
                in_dim: The number of input channels
                n_classes: The number of classes
                depth: The number of convolution blocks in the downsampling and upsampling arms of the U (default: 4)
                n_filters: The number of filters in the first layer (doubles for each downsample) (default: 16)
                drop_prob: The dropout probability for each layer (default: 0.1)
                y_range: The range of values (low, high) to map to in the output (default: None)
        """
        super(UNet, self).__init__()
        # Contracting Path
        self.ds_conv_1 = ConvBlock(in_dim, n_filters)
        self.ds_conv_2 = ConvBlock(n_filters, 2 * n_filters)
        self.ds_conv_3 = ConvBlock(2 * n_filters, 4 * n_filters)
        self.ds_conv_4 = ConvBlock(4 * n_filters, 8 * n_filters)

        self.ds_maxpool_1 = maxpool()
        self.ds_maxpool_2 = maxpool()
        self.ds_maxpool_3 = maxpool()
        self.ds_maxpool_4 = maxpool()
        
        self.ds_dropout_1 = dropout(drop_prob)
        self.ds_dropout_2 = dropout(drop_prob)
        self.ds_dropout_3 = dropout(drop_prob)
        self.ds_dropout_4 = dropout(drop_prob)
        
        self.bridge = ConvBlock(8 * n_filters, 16 * n_filters)
        
        # Expansive Path
        self.us_tconv_4 = TransposeConvBlock(16 * n_filters, 8 * n_filters)
        self.us_tconv_3 = TransposeConvBlock(8 * n_filters, 4 * n_filters)
        self.us_tconv_2 = TransposeConvBlock(4 * n_filters, 2 * n_filters)
        self.us_tconv_1 = TransposeConvBlock(2 * n_filters, n_filters)

        self.us_conv_4 = ConvBlock(16 * n_filters, 8 * n_filters)
        self.us_conv_3 = ConvBlock(8 * n_filters, 4 * n_filters)
        self.us_conv_2 = ConvBlock(4 * n_filters, 2 * n_filters)
        self.us_conv_1 = ConvBlock(2 * n_filters, 1 * n_filters)

        self.us_dropout_4 = dropout(drop_prob)
        self.us_dropout_3 = dropout(drop_prob)
        self.us_dropout_2 = dropout(drop_prob)
        self.us_dropout_1 = dropout(drop_prob)

        self.output = nn.Sequential(nn.Conv2d(n_filters, n_classes, 1), Sigmoid(y_range))

    def forward(self, x):
        """Forward pass.
        
            Args:
                x: The input to the layer
                
            Returns:
                The output from the layer
        """
        res = x

        # Downsample
        res = self.ds_conv_1(res); conv_stack_1 = res.clone()
        res = self.ds_maxpool_1(res)
        res = self.ds_dropout_1(res)
        
        res = self.ds_conv_2(res); conv_stack_2 = res.clone()
        res = self.ds_maxpool_2(res)
        res = self.ds_dropout_2(res)
        
        res = self.ds_conv_3(res); conv_stack_3 = res.clone()
        res = self.ds_maxpool_3(res)
        res = self.ds_dropout_3(res)
        
        res = self.ds_conv_4(res); conv_stack_4 = res.clone()
        res = self.ds_maxpool_4(res)
        res = self.ds_dropout_4(res)
        
        # Bridge
        res = self.bridge(res)
        
        # Upsample
        res = self.us_tconv_4(res)
        res = torch.cat([res, conv_stack_4], dim=1)
        res = self.us_dropout_4(res)
        res = self.us_conv_4(res)

        res = self.us_tconv_3(res)
        res = torch.cat([res, conv_stack_3], dim=1)
        res = self.us_dropout_3(res)
        res = self.us_conv_3(res)
        
        res = self.us_tconv_2(res)
        res = torch.cat([res, conv_stack_2], dim=1)
        res = self.us_dropout_2(res)
        res = self.us_conv_2(res)
        
        res = self.us_tconv_1(res)
        res = torch.cat([res, conv_stack_1], dim=1)
        res = self.us_dropout_1(res)
        res = self.us_conv_1(res)
        
        output = self.output(res)

        return output

In [5]:
# network.py

# from model import *

import numpy as np
import torch
import torch.optim as opt


def set_seed(seed):
    """Set the various seeds and flags to ensure deterministic performance
    
        Args:
            seed: The random seed
    """
    torch.backends.cudnn.deterministic = True   # Note, can impede performance
    torch.backends.cudnn.benchmark = False
    np.random.seed(seed)
    torch.manual_seed(seed)


def get_class_weights(stats):
    """Get the weights for each class
    
        Each class has a weight inversely proportional to the number of instances in the training set
    
        Args:
            stats: The number of instances of each class
        
        Returns:
            The weights for each class
    """
    if np.any(stats == 0.):
        print("Found a class that doesn't appear")
        idx = np.where(stats == 0.)
        stats[idx] = 1
        weights = 1. / stats
        weights[idx] = 0
    else:
        weights = 1. / stats
    return [weight / sum(weights) for weight in weights]


def load_model_only(filename, num_classes, device):
    """Load a model

        Args:
            filename: The name of the file with the pretrained model parameters
            num_classes: The number of classes available to predict
            weights: The weights to apply to the classes
            device: The device on which to run

        Returns:
            A tuple composed (in order) of the model, loss function, and optimiser
    """
    model = UNet(1, n_classes = num_classes, depth = 4, n_filters = 16, y_range = (0, num_classes - 1))
    model.load_state_dict(torch.load(filename, map_location=device))
    model.eval()
    return model


def load_model(filename, num_classes, weights, device):
    """Load a model

        Args:
            filename: The name of the file with the pretrained model parameters
            num_classes: The number of classes available to predict
            weights: The weights to apply to the classes
            device: The device on which to run

        Returns:
            A tuple composed (in order) of the model, loss function, and optimiser
    """
    model = UNet(1, n_classes = num_classes, depth = 4, n_filters = 16, y_range = (0, num_classes - 1))
    model.load_state_dict(torch.load(filename, map_location=device))
    model.eval()
    loss_fn = nn.CrossEntropyLoss(torch.as_tensor(weights, device=device, dtype=torch.float))
    optim = opt.Adam(model.parameters())
    return model, loss_fn, optim


def save_model(model, input, filename):
    """Save the model
    
        The model is saved as both a pkl file and a TorchScript pt file, which can be loaded via
            model.load_state_dict(torch.load(PATH))
            model.eval()
        
        Args:
            model: The model to save
            input: An example input to the model
            filename: The output filename, without file extension
    """
    torch.save(model.state_dict(), f"{filename}.pkl")

    
def accuracy(pred, truth, nearby=False):
    """Get the network accuracy
    
        Args:
            pred: The network prediction
            truth: The true class
            nearby: Whether to consider adjacent classes acceptable
        
        Returns:
            The accuracy
    """
    target = truth.squeeze(1)
    pred_cls = pred.argmax(dim=1)
    mask = (target != 0)
    if nearby:
        result = abs(pred_cls[mask] - target[mask]) <= 1
    else:
        result = pred_cls[mask] == target[mask]
    return result.float().mean()


def create_model(num_classes, weights, device):
    """Create the model

        Args:
            num_classes: The number of classes available to predict
            weights: The weights to apply to the classes
            device: The device on which to run

        Returns:
            A tuple composed (in order) of the model, loss function, and optimiser
    """
    model = UNet(1, n_classes = num_classes, depth = 4, n_filters = 16, y_range = (0, num_classes - 1))
    loss_fn = nn.CrossEntropyLoss(torch.as_tensor(weights, device=device, dtype=torch.float))
    optim = opt.Adam(model.parameters())
    return model, loss_fn, optim


In [10]:
# data.py

import numpy as np
import torch
import os

from collections import OrderedDict
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm


class SegmentationDataset(Dataset):
    """Dataset suitable for segmentation tasks, backed by sharded, compressed .npz files.
 
        Each shard is a single .npz file (written via np.savez_compressed) holding a
        stacked array of many samples under the key 'arr_0'. A `shard_index` array maps a
        global dataset idx to (shard_id, local_idx). Because compressed .npz cannot be
        memory-mapped, each shard is fully decompressed on first access and the resulting
        array is cached per worker process, so the decompression cost is paid once per
        shard per worker rather than once per sample.
 
        The cache is bounded (max_cached_shards) and evicts least-recently-used shards once
        full. Without a bound, a single full epoch eventually decompresses and caches every
        shard the training set touches - with num_workers=0 in particular (e.g. when device
        is set), that means the *entire* dataset ends up decompressed in one process's
        memory by the end of the first epoch, with no eviction. Bounding the cache trades
        some repeated decompression (if the working set revisits an evicted shard) for a
        predictable memory ceiling.
 
        By default tensors are built on CPU, meant to be moved to GPU in the training loop
        with `.to(device, non_blocking=True)` (works together with num_workers>0 + pin_memory
        for overlapped I/O). If `device` is set to a CUDA device here instead, samples are
        placed on the GPU inside __getitem__ directly. CUDA tensors cannot cross process
        boundaries, so this only works with num_workers=0 (single-process, synchronous
        loading) - SegmentationBunch enforces this automatically when device is set.
    """
 
    def __init__(self, image_shards, mask_shards, shard_index, transform=False, device=None,
                 max_cached_shards=8):
        """Constructor.
 
            Args:
                image_shards: Array of shard .npy file paths, each holding a stacked array
                    of image samples. Indexed by shard_id.
                mask_shards: Same shape/order as image_shards, for the corresponding masks.
                shard_index: Array of shape (n_samples, 2) with (shard_id, local_idx) pairs,
                    one row per dataset sample. Build with build_shard_index().
                transform: Whether or not to apply random flip/transpose augmentation
                    (default: False).
                device: If given, samples are created directly on this device (e.g.
                    torch.device('cuda:0')) instead of staying on CPU. Requires
                    num_workers=0 on the DataLoader (default: None -> CPU tensors).
                max_cached_shards: Maximum number of decompressed shard arrays (counting
                    image and mask shards separately) to keep in memory per worker process
                    before evicting the least-recently-used one (default: 8). Higher values
                    mean fewer repeated decompressions but a higher memory ceiling; tune
                    based on shard size and available RAM. With num_workers=0 this cap
                    applies to the single main-process cache; with num_workers>0 each worker
                    keeps its own cache of up to this many shards.
        """
        self.image_shards = image_shards
        self.mask_shards = mask_shards
        self.shard_index = shard_index
        self.transform = transform
        self.device = device
        self.max_cached_shards = max_cached_shards
        self._shard_cache = OrderedDict()  # populated lazily per-worker; not shared across processes
 
    def __len__(self):
        """Retrieve the number of samples in the dataset.
 
            Returns:
                The number of samples in the dataset
        """
        return len(self.shard_index)
 
    def _load_shard(self, shard_id, which):
        """Load a shard file, caching the decompressed array per-worker with LRU eviction.
 
            .npz (savez_compressed) shards cannot be memory-mapped - the whole array must
            be decompressed into memory on first access. The cache means that cost is paid
            once per shard per worker process (not once per sample), bounded to at most
            self.max_cached_shards entries - accessing a cached shard moves it to the
            most-recently-used end; once full, the least-recently-used shard is evicted.
 
            Args:
                shard_id: Index into self.image_shards / self.mask_shards
                which: Either "image" or "mask"
 
            Returns:
                The decompressed array for the requested shard
        """
        key = (shard_id, which)
        if key in self._shard_cache:
            self._shard_cache.move_to_end(key)
            return self._shard_cache[key]
 
        path = self.image_shards[shard_id] if which == "image" else self.mask_shards[shard_id]
        with np.load(path) as npz:
            array = npz['arr_0']
        self._shard_cache[key] = array
        if len(self._shard_cache) > self.max_cached_shards:
            self._shard_cache.popitem(last=False)  # evict least-recently-used
        return array
 
    def __getitem__(self, idx):
        """Retrieve a sample from the dataset.
 
            Args:
                idx: The index of the sample to be retrieved
 
            Returns:
                A (image, mask) tuple of tensors, on self.device if set, else CPU
        """
        shard_id, local_idx = self.shard_index[idx]
        image = np.array(self._load_shard(shard_id, "image")[local_idx])
        mask = np.array(self._load_shard(shard_id, "mask")[local_idx])
 
        image = torch.as_tensor(np.expand_dims(image, axis=0), device=self.device, dtype=torch.float)
        mask = torch.as_tensor(mask, device=self.device, dtype=torch.long)
 
        if self.transform:
            should_hflip = torch.rand(1).item() > 0.5
            should_vflip = torch.rand(1).item() > 0.5
            should_transpose = torch.rand(1).item() > 0.5
            if should_hflip:
                image = torch.flip(image, dims=[-1])
                mask = torch.flip(mask, dims=[-1])
            if should_vflip:
                image = torch.flip(image, dims=[-2])
                mask = torch.flip(mask, dims=[-2])
            if should_transpose:
                image = image.transpose(-2, -1)
                mask = mask.transpose(-2, -1)
 
        return (image, mask)
 
 
def _scan_dir(path):
    """Fast flat-directory file listing (no recursion, no stat-per-subdir like os.walk).
 
        Args:
            path: Directory to scan
 
        Returns:
            A numpy array of filenames (not full paths) found directly in `path`
    """
    with os.scandir(path) as it:
        return np.array([entry.name for entry in it if entry.is_file()])
 
 
def _shard_lengths(shard_paths):
    """Get the number of samples in each shard.
 
        Args:
            shard_paths: List/array of shard .npz file paths (savez_compressed), each
                holding a stacked array under the key 'arr_0'
 
        Returns:
            A numpy array of per-shard sample counts, same order as shard_paths
 
        Note:
            Unlike plain .npy, a compressed .npz has no header that can be read without
            decompressing the array, so this does pay a full decompression cost per shard.
            That cost is paid once, at dataset construction time, not per epoch or per
            sample - the per-worker cache in SegmentationDataset avoids repeating it during
            training.
    """
    lengths = np.empty(len(shard_paths), dtype=np.int64)
    for i, path in enumerate(tqdm(shard_paths, desc='Getting samples in each shard')):
        with np.load(path) as npz:
            lengths[i] = npz['arr_0'].shape[0]
    return lengths
 
 
def build_shard_index(shard_paths, sample_indices):
    """Build a (shard_id, local_idx) index for a subset of samples drawn across shards.
 
        Args:
            shard_paths: List/array of shard .npy file paths (same order used for shard_id)
            sample_indices: Global sample indices (0-based, contiguous across all shards in
                order) to include, e.g. a train or validation split
 
        Returns:
            A numpy array of shape (len(sample_indices), 2) with (shard_id, local_idx) pairs
    """
    lengths = _shard_lengths(shard_paths)
    offsets = np.concatenate([[0], np.cumsum(lengths)])
    shard_index = np.empty((len(sample_indices), 2), dtype=np.int64)
    # searchsorted against cumulative offsets locates the owning shard for each global index
    shard_ids = np.searchsorted(offsets, sample_indices, side='right') - 1
    shard_index[:, 0] = shard_ids
    shard_index[:, 1] = sample_indices - offsets[shard_ids]
    return shard_index
 
 
class SegmentationBunch():
    """Associates batches of training and validation datasets suitable for segmentation
        tasks, reading from the sharded .npy output of preprocess.py (shard_0.npy,
        shard_1.npy, ... per class directory - not one file per sample).
    """
 
    def __init__(self, root_dir, balance_map, image_dir, mask_dir, batch_size, valid_pct=0.1,
                 test_pct=0.0, transform=False, num_workers=8, pin_memory=True, device=None,
                 max_cached_shards=8):
        """Constructor.
 
            Args:
                root_dir: The top-level directory containing the shard files
                balance_map: Dict mapping the different classes (NuMI/numu, NuMI/nue, etc...)
                    to their directory and target fraction of the total, e.g.:
                        {
                            'class_1': {'dir': 'NuMI/numu', 'fraction': 0.25},
                            'class_2': {'dir': 'BNB/numu', 'fraction': 0.5},
                            'class_3': {'dir': 'BNB/nue', 'fraction': 0.25}
                        }
                    Fractions must sum to 1.0. Sizes are normalized to the bottleneck class.
                image_dir: The relative directory (under each class dir) containing image shards
                mask_dir: The relative directory (under each class dir) containing mask shards
                batch_size: The batch size
                valid_pct: The fraction of samples to be used for validation (default: 0.1)
                test_pct: The fraction of samples reserved for testing, currently unused for
                    splitting but kept to preserve the (valid_pct + test_pct) < 1 sanity check
                    (default: 0.0)
                transform: Whether or not to apply augmentation to the training set (default: False)
                num_workers: DataLoader worker processes for parallel I/O (default: 8).
                    Ignored (forced to 0) if device is set - see device below.
                pin_memory: Whether to use pinned host memory so H2D copies can be async
                    (default: True). Ignored (forced to False) if device is set, since pinned
                    memory only matters for CPU tensors being copied to GPU.
                device: If given (e.g. torch.device('cuda:0')), samples are created directly
                    on this device inside the Dataset instead of staying on CPU. CUDA tensors
                    cannot cross process boundaries, so this forces num_workers=0 and
                    pin_memory=False - loading becomes single-process and synchronous, and
                    each __getitem__ call blocks the GPU until the sample is ready. This
                    trades away the parallel-I/O speedup for simplicity (no explicit
                    .to(device) needed in the training loop). Default: None -> CPU tensors,
                    parallel loading via num_workers.
                max_cached_shards: Maximum decompressed shards kept per worker's cache before
                    LRU eviction kicks in (default: 8). See SegmentationDataset for details.
                    With num_workers=0 (including whenever device is set), this bounds the
                    single main-process cache directly - important since otherwise a full
                    epoch would decompress and retain every shard in one process's memory.
        """
        assert (valid_pct + test_pct) < 1.
        total_fraction = sum(cfg['fraction'] for cfg in balance_map.values())
        assert np.isclose(total_fraction, 1.0), "Fractions in balance_map must sum to 1.0"
 
        if device is not None:
            if num_workers > 0:
                print(f"device={device} was set: forcing num_workers=0, pin_memory=False and shuffle_training=False"
                      f"(was {num_workers}) since CUDA tensors can't cross process boundaries")
            num_workers = 0
            pin_memory = False
            shuffle_training = False
 
        per_class_shard_index = {}
        per_class_img_shards = {}
        per_class_msk_shards = {}
        per_class_count = {}
 
        # 1. Index every class's shards (cheap - only reads .npy headers, not data)
        for class_name, config in tqdm(balance_map.items(), desc='Indexing shards'):
            img_shard_dir = os.path.join(root_dir, config['dir'], image_dir)
            msk_shard_dir = os.path.join(root_dir, config['dir'], mask_dir)

            img_shard_names = set(_scan_dir(img_shard_dir).tolist())
            msk_shard_names = set(_scan_dir(msk_shard_dir).tolist())
 
            if img_shard_names != msk_shard_names:
                # Fail here, at construction time, rather than deep inside a DataLoader
                # worker mid-epoch. This mismatch almost always means a preprocessing run
                # was interrupted (e.g. disk quota/full) between writing a Hits shard and
                # its matching Truth shard - the fix is to remove the orphaned shard(s) and
                # re-run preprocessing for the affected file, not a code change here.
                only_in_img = sorted(img_shard_names - msk_shard_names)
                only_in_msk = sorted(msk_shard_names - img_shard_names)
                msg = [f"Hits/Truth shard mismatch for class '{class_name}' ({config['dir']}):"]
                if only_in_img:
                    msg.append(f"  present in {image_dir} but missing from {mask_dir}: {only_in_img}")
                if only_in_msk:
                    msg.append(f"  present in {mask_dir} but missing from {image_dir}: {only_in_msk}")
                msg.append(
                    "  This usually means preprocessing was interrupted partway through a "
                    "flush() - remove the orphaned shard file(s) above and re-run "
                    "preprocessing for this class before retrying."
                )
                raise FileNotFoundError("\n".join(msg))
 
            img_shard_names = sorted(_scan_dir(img_shard_dir))
            img_shards = np.array([os.path.join(img_shard_dir, n) for n in tqdm(img_shard_names, desc=f'Indexing images for class {class_name}')])
            msk_shards = np.array([os.path.join(msk_shard_dir, n) for n in tqdm(img_shard_names, desc=f'Indexing masks for class {class_name}')])
 
            lengths = _shard_lengths(img_shards)
            per_class_shard_index[class_name] = build_shard_index(img_shards, np.arange(lengths.sum()))
            per_class_img_shards[class_name] = img_shards
            per_class_msk_shards[class_name] = msk_shards
            per_class_count[class_name] = lengths.sum()
 
        # 2. Find the total dataset capacity dictated by the bottleneck class
        max_total = min(per_class_count[c] / balance_map[c]['fraction'] for c in balance_map)
 
        train_rows, valid_rows = [], []
        shard_paths_img, shard_paths_msk = [], []
        shard_offset = 0
 
        # 3. Sample exactly what we need from each class to preserve ratios, then split
        for class_name, config in tqdm(balance_map.items(), desc='Sampling classes'):
            n_to_sample = int(max_total * config['fraction'])
            full_index = per_class_shard_index[class_name]
            chosen = full_index[np.random.permutation(len(full_index))[:n_to_sample]].copy()
 
            # Re-map local shard ids to a single global shard list shared across classes
            chosen[:, 0] += shard_offset
            shard_paths_img.append(per_class_img_shards[class_name])
            shard_paths_msk.append(per_class_msk_shards[class_name])
            shard_offset += len(per_class_img_shards[class_name])
 
            n_valid = int(len(chosen) * valid_pct)
            perm = np.random.permutation(len(chosen))
            valid_rows.append(chosen[perm[:n_valid]])
            train_rows.append(chosen[perm[n_valid:]])
 
        img_shards_all = np.concatenate(shard_paths_img)
        msk_shards_all = np.concatenate(shard_paths_msk)
        train_index = np.concatenate(train_rows)
        valid_index = np.concatenate(valid_rows)
        np.random.shuffle(train_index)
 
        train_ds = SegmentationDataset(img_shards_all, msk_shards_all, train_index, transform=transform, 
                                       device=device, max_cached_shards=max_cached_shards)
        valid_ds = SegmentationDataset(img_shards_all, msk_shards_all, valid_index, transform=False, 
                                       device=device, max_cached_shards=max_cached_shards)
 
        self.train_dl = DataLoader(
            train_ds, batch_size=batch_size, shuffle=shuffle_training, drop_last=True,
            num_workers=num_workers, pin_memory=pin_memory,
            persistent_workers=(num_workers > 0), prefetch_factor=4 if num_workers > 0 else None
        )
        self.valid_dl = DataLoader(
            valid_ds, batch_size=batch_size, shuffle=False, drop_last=True,
            num_workers=num_workers, pin_memory=pin_memory,
            persistent_workers=(num_workers > 0), prefetch_factor=4 if num_workers > 0 else None
        )

        self.device = device
 
    def count_classes(self, num_classes):
        """Count the number of instances of each class in the training set.
 
            Args:
                num_classes: The number of classes in the training set
 
            Returns:
                A numpy array of the number of instances of each class
        """

        ds = self.train_dl.dataset
        device = self.device
 
        # Group this Dataset's training sample indices by which shard they live in, so
        # each mask shard file is opened and decompressed exactly once.
        shard_to_local = {}
        for shard_id, local_idx in ds.shard_index:
            shard_to_local.setdefault(int(shard_id), []).append(int(local_idx))
 
        count = torch.zeros(num_classes, dtype=torch.long, device=device)
        for shard_id, local_indices in tqdm(shard_to_local.items(), desc='Counting classes'):
            with np.load(ds.mask_shards[shard_id]) as npz:
                # Fancy-index only the rows that belong to the training split - a shard can
                # contain both train and validation rows - then let the full decompressed
                # shard array go out of scope and be freed at the end of this iteration.
                truth = torch.from_numpy(npz['arr_0'][local_indices]).long()
                if device is not None:
                    truth = truth.to(device, non_blocking=True)
                count += torch.bincount(truth.flatten(), minlength=num_classes)
        return count.cpu().numpy()

        
        # count = torch.zeros(num_classes, dtype=torch.long, device=device)
        # for _, truth in tqdm(self.train_dl, desc='Counting classes'):
        #     if device is not None:
        #         truth = truth.to(device, non_blocking=True)
        #     count += torch.bincount(truth.flatten(), minlength=num_classes)
        # return count.cpu().numpy()

In [11]:
# # data.py

# import numpy as np
# import torch
# import os
# from torch.utils.data import Dataset
# from torch.utils.data import DataLoader


# class SegmentationDataset(Dataset):
#     """Dataset suitable for segmentation tasks.
#     """

#     # def __init__(self, image_dir, mask_dir, filenames, transform=False, device=torch.device('cuda:0')):
#     #     """Constructor.

#     #         Args:
#     #             image_dir: The directory containing the images
#     #             mask_dir: The directory containing the masks
#     #             filenames: The filanems for the images associate with this dataset
#     #             transform: Whether or not to transform the items (default: False).
#     #             device: The device on which tensors should be created (default: 'cuda:0')
#     #     """
#     #     self.image_dir = image_dir
#     #     self.mask_dir = mask_dir
#     #     self.transform = transform
#     #     self.filenames = filenames
#     #     self.device = device

#     def __init__(self, image_paths, mask_paths, transform=False, device=torch.device('cuda:0')):
#         """Constructor.

#             Args:
#                 image_paths: Array/list of absolute or resolved full paths to the image files.
#                 mask_paths: Array/list of absolute or resolved full paths to the mask files.
#                 transform: Whether or not to transform the items (default: False).
#                 device: The device on which tensors should be created (default: 'cuda:0')
#         """
#         assert len(image_paths) == len(mask_paths), "Mismatched number of images and masks!"
#         self.image_paths = image_paths
#         self.mask_paths = mask_paths
#         self.transform = transform
#         self.device = device


#     def __len__(self):
#         """Retrieve the number of samples in the dataset.
        
#             Returns:
#                 The number of samples in the dataset
#         """
#         # return len(self.filenames)
#         return len(self.image_paths)


#     def __getitem__(self, idx):
#         """Retrieve a sample from the dataset.
        
#             Args:
#                 idx: The index of the sample to be retrieved
        
#             Returns:
#                 The sample requested
#         """
#         # img_name = os.path.join(self.image_dir, self.filenames[idx])
#         # with open(img_name, 'rb') as file:
#         #     image = np.load(file)['arr_0']
        
#         # mask_name = os.path.join(self.mask_dir, self.filenames[idx])
#         # with open(mask_name, 'rb') as file:
#         #     mask = np.load(file)['arr_0']

#         # Directly use the pre-built full paths
#         img_name = self.image_paths[idx]
#         with open(img_name, 'rb') as file:
#             image = np.load(file)['arr_0']
        
#         mask_name = self.mask_paths[idx]
#         with open(mask_name, 'rb') as file:
#             mask = np.load(file)['arr_0']
                
#         image = torch.as_tensor(np.expand_dims(image, axis=0), device=self.device, dtype=torch.float)
#         mask = torch.as_tensor(mask, device=self.device, dtype=torch.long)
        
#         if self.transform:
#             should_hflip = True if torch.rand(1) > 0.5 else False
#             should_vflip = True if torch.rand(1) > 0.5 else False
#             should_transpose = True if torch.rand(1) > 0.5 else False
#             # need to check that these make sense in the context of pixel classification
#             if should_hflip:
#                 image = tv.transforms.functional.hflip(image)
#                 mask = tv.transforms.functional.hflip(mask)
#             if should_vflip:
#                 image = tv.transforms.functional.vflip(image)
#                 mask = tv.transforms.functional.vflip(mask)
#             if should_transpose:
#                 image = image.transpose(1, 2)
#                 mask = mask.transpose(1, 2)
        
#         return (image, mask)


# class SegmentationBunch():
#     """Associates batches of training, validation and testing datasets suitable
#         for segmentation tasks.
#     """
    
#     def __init__(self, root_dir, balance_map, image_dir, mask_dir, batch_size, train_pct=None, valid_pct=0.1,
#                  test_pct=0.0, transform=False, device=torch.device('cuda:0')):
#         """Constructor.

#             Args:
#                 root_dir: The top-level directory containing the images
#                 balance_map: Dict mapping the different classes (NuMI/numu, NuMI/nue, etc...) assigning 
#                     the correct fration w.r.t the total (i.e., 
#                         {
#                             'class_1': {'dir': 'NuMI/numu', 'fraction': 0.25}, 
#                             'class_2': {'dir': 'BNB/numu', 'fraction': 0.5},
#                             'class_3': {'dir': 'BNB/nue', 'fraction': 0.25}
#                         } 
#                     would mean having half BNB/numu, then a quarter for each other sample; size are normalized 
#                     to smallest sample)
#                 image_dir: The relative directory containing the images
#                 mask_dir: The relative directory containing the masks
#                 batch_size: The batch size
#                 valid_pct: The fraction of images to be used for validation (default: 0.1)
#                 test_pct: The fraction of images to be used for testing (default: 0.0)
#                 transform: Whether or not to transform the items (default: False)
#                 device: The device on which tensors should be created (default: 'cuda:0')
#         """
#         assert((valid_pct + test_pct) < 1.)
#         # Ensure fractions sum to 1.0 (or close to it due to floats)
#         total_fraction = sum(cfg['fraction'] for cfg in balance_map.values())
#         assert np.isclose(total_fraction, 1.0), "Fractions in balance_map must sum to 1.0"
        
#         # image_dir = os.path.join(root_dir, image_dir)
#         # mask_dir = os.path.join(root_dir, mask_dir)
#         # image_filenames = np.array(next(os.walk(image_dir))[2])
#         # print(image_filenames)
#         # n_files = len(image_filenames)
#         # valid_size = int(n_files * valid_pct)
#         # train_size = n_files - valid_size if train_pct is None else int(n_files * train_pct)
        
#         # sample = np.random.permutation(n_files)
#         # train_sample = sample[valid_size:] if not train_size else \
#         #     sample[valid_size:valid_size + train_size]
#         # valid_sample = sample[:valid_size]
                
#         # train_ds = SegmentationDataset(image_dir, mask_dir, image_filenames[train_sample], transform, device)
#         # self.train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=False, drop_last=True, num_workers=0)
        
#         # valid_ds = SegmentationDataset(image_dir, mask_dir, image_filenames[valid_sample], None, device)
#         # self.valid_dl = DataLoader(valid_ds, batch_size=batch_size, shuffle=False, drop_last=True, num_workers=0)

#         class_files = {}
        
#         # 1. Scan directories and gather available file counts
#         for class_name, config in balance_map.items():
#             # Build paths: root_dir + dir + image_dir / mask_dir
#             class_img_dir = os.path.join(root_dir, config['dir'], image_dir)
#             class_msk_dir = os.path.join(root_dir, config['dir'], mask_dir)
            
#             # Grab all image filenames
#             filenames = np.array(next(os.walk(class_img_dir))[2])
            
#             class_files[class_name] = {
#                 'filenames': filenames,
#                 'image_dir': class_img_dir,
#                 'mask_dir': class_msk_dir,
#                 'count': len(filenames),
#                 'fraction': config['fraction']
#             }

#         # 2. Find the total dataset capacity dictated by the bottleneck class
#         # formula: total = min(available_files / target_fraction)
#         max_total = min(cls_info['count'] / cls_info['fraction'] for cls_info in tqdm(class_files.values(), desc=f'Loading max'))
        
#         all_img_paths = []
#         all_msk_paths = []
        
#         # 3. Sample exactly what we need from each class to preserve ratios
#         for class_name, cls_info in tqdm(class_files.items(), desc='Sampling classes'):
#             n_to_sample = int(max_total * cls_info['fraction'])
            
#             # Shuffle indices to pick a random subset of files from this class
#             shuffled_indices = np.random.permutation(cls_info['count'])[:n_to_sample]
#             sampled_filenames = cls_info['filenames'][shuffled_indices]
            
#             # Build individual absolute file paths
#             for fname in tqdm(sampled_filenames, desc=f'For class {class_name}, adding files'):
#                 all_img_paths.append(os.path.join(cls_info['image_dir'], fname))
#                 all_msk_paths.append(os.path.join(cls_info['mask_dir'], fname))

#         # Convert to arrays for clean numpy slicing
#         all_img_paths = np.array(all_img_paths)
#         all_msk_paths = np.array(all_msk_paths)
        
#         # 4. Train / Validation splitting
#         n_total_samples = len(all_img_paths)
#         valid_size = int(n_total_samples * valid_pct)
#         train_size = n_total_samples - valid_size if train_pct is None else int(n_total_samples * train_pct)
        
#         # Global shuffle so classes are thoroughly mixed across batches
#         global_indices = np.random.permutation(n_total_samples)
        
#         train_sample = global_indices[valid_size:] if not train_size else \
#             global_indices[valid_size:valid_size + train_size]
#         valid_sample = global_indices[:valid_size]
                
#         # NOTE: Your SegmentationDataset needs to accept lists of full paths instead of directory prefixes
#         train_ds = SegmentationDataset(all_img_paths[train_sample], all_msk_paths[train_sample], transform, device)
#         self.train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=False, drop_last=True, num_workers=0)
#         print('DEBUG: Created train_dl')
        
#         valid_ds = SegmentationDataset(all_img_paths[valid_sample], all_msk_paths[valid_sample], None, device)
#         self.valid_dl = DataLoader(valid_ds, batch_size=batch_size, shuffle=False, drop_last=True, num_workers=0)
#         print('DEBUG: Created train_dl')


#     def count_classes(self, num_classes):
#         """Count the number of instances of each class in the training set
        
#             Args:
#                 num_classes: The number of classes in the training set
                
#             Returns:
#                 A list of the number of instances of each class
#         """
#         # count = np.zeros(num_classes)
#         # for batch in tqdm(self.train_dl, desc='Counting classes'):
#         #     _, truth = batch
#         #     unique, counts = torch.unique(truth, return_counts=True)
#         #     unique = [ u.item() for u in unique ]
#         #     counts = [ c.item() for c in counts ]
#         #     this_dict = dict(zip(unique, counts))
#         #     for key in this_dict:
#         #         count[key] += this_dict[key]
#         # return count

#         count = None
    
#         for batch in tqdm(self.train_dl, desc='Counting classes'):
#             _, truth = batch
            
#             # Initialize count on the same device as the data on the first iteration
#             if count is None:
#                 count = torch.zeros(num_classes, dtype=torch.long, device=truth.device)
                
#             # Everything stays on the same device (GPU) for maximum speed
#             count += torch.bincount(truth.flatten(), minlength=num_classes)
            
#         # Move to CPU only at the very end to convert to a NumPy array
#         return count.cpu().numpy()

#     # def count_classes(self, num_classes):
#     #     """Count the number of instances of each class directly from mask files, 
#     #     bypassing image loading, transforms, and data loader overhead.
#     #     """
#     #     # Grab the list of mask paths directly from the underlying dataset
#     #     mask_paths = self.train_dl.dataset.mask_paths
        
#     #     # Keep the count on the CPU using a standard torch tensor
#     #     count = torch.zeros(num_classes, dtype=torch.long)
        
#     #     print("Running fast mask-only class count...")
#     #     for mask_path in tqdm(mask_paths, desc='Counting classes'):
#     #         # 1. Only open the mask file
#     #         with open(mask_path, 'rb') as file:
#     #             mask_np = np.load(file)['arr_0']
                
#     #         # 2. Convert to CPU tensor and count immediately
#     #         mask_tensor = torch.as_tensor(mask_np, dtype=torch.long)
#     #         count += torch.bincount(mask_tensor.flatten(), minlength=num_classes)
            
#     #     return count.numpy()

# Run network training here

The key parameters that will need editing are the view and the pass to be trained. Each view/pass combination has its own network. Views are specified using the standard U, V, W nomenclature (and is consistent with the file naming conventions from previous steps), while the pass is either pass 1 or pass 2.

The respective variables can be set in the cell below via <code>view</code> and <code>vertex_pass</code>.

If you edited the <code>thresholds</code> variable at the <code>make_images</code> stage, you may need to update the <code>NUM_CLASSES</code> variable to reflect any change in the number of thresholds. Note that this value should be equal to the length of the <code>thresholds</code> variable, despite this variable specifying bin edges, because one extra class is required to represent the null case where a pixel has no hits in it.

<code>batch_size</code> can, of course, be varied according to the available resources of your GPU, but as a semantic segmentation network you'll need a lot of memory on your GPU to increase this beyond the current default of 32.

The <code>image_path</code> variable should contain the path to the images generated by the <code>make_images</code> notebook (i.e. <code>global_path</code>), and will expect to find the <code>Hits</code> and <code>Truth</code> folders within that path).

> Note that the code has been updated to allow for sample balancing: taking as input the `balance_map` dictionary, it can, for each sample class, consider a different fraction.
> The dict. is required to follow this schema
> ```python
> {
>   'class_1': {'dir': 'path_1', 'fraction': 0.75},
>   'class_2': {'dir': 'path_2', 'fraction': 0.25}
> }
> ```

Note that the cells below will count the class representation in the training set, determine how to weight them and print this out. It's worth taking a look at this output to ensure all classes are represented, as training will fail if they are not.

You will want to set the number of epochs, <code>n_epochs</code> to train for. This is not easy to determine a priori, but 20 is a reasonable starting point (plots of the loss function and accuracy are produced to help you determine when the network has effectively trained).

<code>model_name</code> acts as a prefix for saving the model. The state of the model is saved after every epoch, with a suffix indicating the epoch number.

Once you are happy with the variable values you can run all of the cells in this section in order (having run all of the cells above), with the final cell in this section actually performing the training.

# First training: perfectly balanced 50-50/50-50 samples

In [12]:
# This line is important for GPU running, otherwise some weights end up on the CPU
torch.set_default_tensor_type(torch.cuda.FloatTensor)

view = "W"
vertex_pass = 1
the_seed = 42
gpu = torch.device('cuda:0')
batch_size=32
NUM_CLASSES = 20   # NULL = 0, various distance bands 1-19
# image_path = f"Accel/Pass{vertex_pass}/Images{view}"
# image_path = '/exp/icarus/data/users/msotgia/vertexStudies/forTraining/ICARUS_DlVertex'
image_path = '/home/msotgia/vertexOnEaf/ICARUS_DlVertex_fastLoader' # For processing on EAF w/o overhead

balance_map = {
    'numi_numu': {'dir': f'NuMI/numu/Pass{vertex_pass}/Images{view}', 'fraction': 0.25},
    'numi_nue':  {'dir': f'NuMI/nue/Pass{vertex_pass}/Images{view}',  'fraction': 0.25},
    'bnb_numu':  {'dir': f'BNB/numu/Pass{vertex_pass}/Images{view}',  'fraction': 0.25},
    'bnb_nue':   {'dir': f'BNB/nue/Pass{vertex_pass}/Images{view}',   'fraction': 0.25}
}

n_epochs = 20
model_name = "icarus_pass1_fully_balanced_beam_flavour"

for subdir in ["models", "stats", "images"]:
    dir = f"outputs/{subdir}/pass{vertex_pass}/{view}"
    if not os.path.exists(dir):
        os.makedirs(dir)

In [13]:
# main.py

#from data import *
#from network import *

set_seed(the_seed)
bunch = SegmentationBunch(image_path, balance_map, "Hits", "Truth", batch_size=batch_size, valid_pct = 0.25, device=gpu)
train_stats = bunch.count_classes(NUM_CLASSES)
weights = get_class_weights(train_stats)

device=cuda:0 was set: forcing num_workers=0, pin_memory=False and shuffle_training=False(was 8) since CUDA tensors can't cross process boundaries


Indexing shards:   0%|          | 0/4 [00:00<?, ?it/s]

Indexing images for class numi_numu:   0%|          | 0/12 [00:00<?, ?it/s]

Indexing masks for class numi_numu:   0%|          | 0/12 [00:00<?, ?it/s]

Getting samples in each shard:   0%|          | 0/12 [00:00<?, ?it/s]

...

Indexing masks for class bnb_nue:   0%|          | 0/16 [00:00<?, ?it/s]

Getting samples in each shard:   0%|          | 0/16 [00:00<?, ?it/s]

Getting samples in each shard:   0%|          | 0/16 [00:00<?, ?it/s]

Sampling classes:   0%|          | 0/4 [00:00<?, ?it/s]

Counting classes:   0%|          | 0/67 [00:00<?, ?it/s]

`weights` before the changes were

```python
[np.float64(8.623750719476234e-08),
 np.float64(0.007172987689688895),
 np.float64(0.0007584045062301771),
 np.float64(0.00037038372234998225),
 np.float64(0.00024288647877137123),
 np.float64(0.000198978541093642),
 np.float64(0.00019057576343509776),
 np.float64(0.0001858506259661026),
 np.float64(0.0003745255916270518),
 np.float64(0.000288019790190076),
 np.float64(0.00048425659795097424),
 np.float64(0.0007499850008340051),
 np.float64(0.001114525094378949),
 np.float64(0.0016114242089710934),
 np.float64(0.0016443771616277915),
 np.float64(0.002822851947461105),
 np.float64(0.004856441516708472),
 np.float64(0.011101357805652648),
 np.float64(0.08215330262912612),
 np.float64(0.8836787790904292)]
 ```

 Below is after 

In [14]:
weights

[np.float64(8.519963684944396e-08),
 np.float64(0.007085679145877703),
 np.float64(0.0007497415625367602),
 np.float64(0.0003665544958584882),
 np.float64(0.00024035761796248665),
 np.float64(0.0001968197952747986),
 np.float64(0.00018825517951914398),
 np.float64(0.00018342249048324701),
 np.float64(0.0003691854737498892),
 np.float64(0.0002839740505565021),
 np.float64(0.000477735292525849),
 np.float64(0.0007437624048099355),
 np.float64(0.0011049789666749603),
 np.float64(0.0015923534020969112),
 np.float64(0.0016293678343016775),
 np.float64(0.0028040317774834034),
 np.float64(0.0048076004793971595),
 np.float64(0.010931800972717272),
 np.float64(0.08072956371002772),
 np.float64(0.8855147301485092)]

In [15]:
np.savez(f'outputs/stats/pass{vertex_pass}/{view}/weights_{model_name}.npz', weights)

In [16]:
train_losses = torch.zeros(n_epochs * len(bunch.train_dl), device=gpu)
val_losses = torch.zeros(n_epochs, device=gpu)
batch_losses = torch.zeros(len(bunch.valid_dl), device=gpu)

train_accs = torch.zeros(n_epochs * len(bunch.train_dl), device=gpu)
val_accs = torch.zeros(n_epochs, device=gpu)
batch_accs = torch.zeros(len(bunch.valid_dl), device=gpu)

In [17]:
# Standard model creation
model, loss_fn, optim = create_model(NUM_CLASSES, weights, gpu)

i = 0
start = 0
finish = n_epochs

In [ ]:
from tqdm.notebook import tqdm
set_seed(the_seed)
for e in tqdm(range(start, finish), desc='Training'):
    model = model.train()
    n_batches = len(bunch.train_dl)
    for b, batch in enumerate(tqdm(bunch.train_dl, desc=f'For epoch {e}, training over batches')):
        x, y = batch
        pred = model.forward(x)
        loss = loss_fn(pred, y)

        train_losses[i] = loss.item()
        train_accs[i] = accuracy(pred, y, nearby=False)

        loss.backward()
        optim.step()
        #scheduler.step()
        optim.zero_grad()
        i += 1
        if b == (n_batches - 1):
            save_model(model, x, f"outputs/models/pass{vertex_pass}/{view}/{model_name}_{e}")

    # Validate
    model = model.eval()
    with torch.no_grad():
        for b, batch in enumerate(tqdm(bunch.valid_dl, desc=f'For epoch {e}, validating over batches')):
            x, y = batch
            pred = model.forward(x)
            loss = loss_fn(pred, y)
            
            batch_losses[b] = loss.item()
            batch_accs[b] = accuracy(pred, y, nearby=False)
        val_losses[e] = torch.mean(batch_losses)
        val_accs[e] = torch.mean(batch_accs)

    np.savez(f'outputs/stats/pass{vertex_pass}/{view}/losses_{model_name}_{e}.npz', 
             train_losses.cpu(), val_losses.cpu(), batch_losses.cpu(), train_accs.cpu(), val_accs.cpu(), batch_accs.cpu())

    

Training:   0%|          | 0/20 [00:00<?, ?it/s]

For epoch 0, training over batches:   0%|          | 0/11190 [00:00<?, ?it/s]

# Assess network performance

Having trained a network, you can look at its performance - this should be run immediately after the network has finished training. The cells below should not require any editing.

The first cell runs over a single batch from the validation set and produuces images allowing you to compare the truth (left image) to the network classification (right image), though it is worth noting that the <code>show_batch</code> function produces all images in the same folder and does not uniquely identify the view or pass, so if you want to retain them, you'll want to move them between runs.

The next three cells produce plots showing the evolution of the network across epochs. Ideally you want to see a plateauing of the loss and accuracy to establish a well trained model, with no evidence that the training and validation performance are diverging (you can always select a model from an earlier epoch before divergence if the network appears to be overfitting - or get more training samples if the network is not adequately trained).

The final cell in this section simply saves the evolution history of the network to allow easy plot regenertion if needed.

In [ ]:
set_seed(the_seed)
model = model.eval()
with torch.no_grad():
    for b, batch in enumerate(bunch.valid_dl):
        x, y = batch
        pred = model.forward(x)
        show_batch(finish, b, x, pred, y, n=32, randomize=False)
        break

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams['lines.linewidth'] = 3
mpl.rcParams['axes.titlesize'] = 32
mpl.rcParams['axes.labelsize'] = 32
mpl.rcParams['xtick.labelsize'] = 22
mpl.rcParams['ytick.labelsize'] = 22
mpl.rcParams['legend.fontsize'] = 24

In [ ]:
fig = plt.figure(figsize=(20, 15))
plt.xlabel('epoch')
plt.ylabel('metric')

tl = torch.mean(train_losses.reshape([n_epochs, -1]), axis=1).detach().cpu()
vl = val_losses.detach().cpu()
plt.plot(tl, label="training loss")
plt.plot(vl, label="validation loss")

plt.legend()

fig.savefig(f"outputs/stats/pass{vertex_pass}/{view}/stats_loss_{vertex_pass}_{view}.pdf", dpi=200)
fig.savefig(f"outputs/stats/pass{vertex_pass}/{view}/stats_loss_{vertex_pass}_{view}.png", dpi=200, facecolor='w')

In [ ]:
fig = plt.figure(figsize=(20, 15))
plt.xlabel('epoch')
plt.ylabel('metric')

ta = torch.mean(train_accs.reshape([n_epochs, -1]), axis=1).detach().cpu()
va = val_accs.detach().cpu()
plt.plot(ta, label="training accuracy")
plt.plot(va, label="validation accuracy")

plt.legend()

fig.savefig(f"outputs/stats/pass{vertex_pass}/{view}/stats_acc_{vertex_pass}_{view}.pdf", dpi=200)
fig.savefig(f"outputs/stats/pass{vertex_pass}/{view}/stats_acc_{vertex_pass}_{view}.png", dpi=200, facecolor='w')

In [ ]:
with open(f'outputs/stats/pass{vertex_pass}/{view}/train_loss_{vertex_pass}_{view}_20.npy', 'wb') as f:
    np.save(f, tl)
with open(f'outputs/stats/pass{vertex_pass}/{view}/val_loss_{vertex_pass}_{view}_20.npy', 'wb') as f:
    np.save(f, vl)
with open(f'outputs/stats/pass{vertex_pass}/{view}/train_accs_{vertex_pass}_{view}_20.npy', 'wb') as f:
    np.save(f, ta)
with open(f'outputs/stats/pass{vertex_pass}/{view}/val_accs_{vertex_pass}_{view}_20.npy', 'wb') as f:
    np.save(f, va)

# Generating a TorchScript network

The network was trained on a GPU, but ultimately runs on a CPU in a C++ context. This means that the network must be converted to TorchScript format. This can be performed using the cell below and can be run at any time - it need not be run immediately after training the network, because it only requires access to a saved model state.

The only parameters requiring editing here are the location of the input file; which is the save model from the chosen training epoch (so some combination of the <code>moidel_name</code> and epoch with a <code>.pkl</code> extension), the <code>output_filename</code>, which should have a <code>.pt</code> extension, and also the number of classes <code>NUM_CLASSES</code>, which should, of course, match the previouslyt specified value.

The resultant <code>.pt</code> files are what will ultimately be loaded by Pandora for network inference.

In [ ]:
# This line is important for ensuring all tensors exist on the same device
torch.set_default_tensor_type(torch.FloatTensor)

filename = f"outputs/models/pass{vertex_pass}/{view}/dunefd_hd_accel_19.pkl"
output_filename = f"PandoraUnet_Vertex_DUNEFD_Accel_{vertex_pass}_{view}.pt"
the_seed = 42
device = torch.device('cpu')
NUM_CLASSES = 20

set_seed(the_seed)

model = load_model_only(filename, NUM_CLASSES, device)

sm = torch.jit.script(model)
sm.save(output_filename)

# Confusion Matrices

In [ ]:
# This line is important for GPU running, otherwise some weights end up on the CPU
torch.set_default_tensor_type(torch.cuda.FloatTensor)

view = "W"
vertex_pass = 1
the_seed = 42
gpu = torch.device('cuda:0')
batch_size=1
NUM_CLASSES = 20   # NULL = 0, various distance bands 1-19
image_path = f"Accel/Pass{vertex_pass}/Images{view}"

for subdir in ["models", "stats", "images"]:
    dir = f"outputs/{subdir}/pass{vertex_pass}/{view}"
    if not os.path.exists(dir):
        os.makedirs(dir)

set_seed(the_seed)
bunch = SegmentationBunch(image_path, "Hits", "Truth", batch_size=batch_size, valid_pct = 0.25, device=gpu)

filename = f"outputs/models/pass{vertex_pass}/{view}/dunefd_hd_accel_19.pkl"
NUM_CLASSES = 20

set_seed(the_seed)

model = load_model_only(filename, NUM_CLASSES, gpu)

In [ ]:
import scipy.stats as stats
binning = np.linspace(0, 20, 21, dtype=int)

model = model.to(gpu)
confusion = np.zeros((20,20))
for img, cls in bunch.valid_dl:
    img = img.to(gpu)
    output = model(img)
    _, preds = torch.max(output, 1)
    
    cls_detached = cls.cpu().numpy().flatten()
    preds_detached = preds.cpu().numpy().flatten()
    
    H, *_ = stats.binned_statistic_2d(preds_detached, cls_detached, None,
                                      bins=[binning, binning], statistic='count')
    confusion += H

In [ ]:
temporary = confusion.copy()
fig = plt.figure(figsize=(15, 10))
plt.xlabel('true class')
plt.ylabel('fraction')
plt.step(list(np.arange(1, 20)), np.sum(temporary[1:], axis=1) / np.sum(temporary[1:]), where="mid")

save_figure(fig, f"outputs/stats/pass{vertex_pass}/{view}/true_class_dist_{vertex_pass}_{view}")

In [ ]:
sums = np.sum(confusion, axis=1).repeat(20).reshape((20,20))
confusion /= sums

In [ ]:
print(f"--- Class Accuracy")
for t in range(confusion.shape[0]):
    print(f"{t:2}: {100*(confusion[t,t] / confusion[t].sum()):.1f}")
print()

In [ ]:
confusion[0,:] = 0
confusion[:,0] = 0

sums = np.sum(confusion, axis=1).repeat(20).reshape((20,20))
sums[0,:] = 1
confusion /= sums

print(f"--- Class Accuracy")
for t in range(confusion.shape[0]):
    print(f"{t:2}: {100*confusion[t,t]:.1f}")
print()

In [ ]:
fig = plt.figure(figsize=(15, 15))
plt.xlabel('truth')
plt.ylabel('network')
plt.imshow(confusion)
plt.colorbar()
save_figure(fig, f"outputs/stats/pass{vertex_pass}/{view}/confusion_{vertex_pass}_{view}")

for t in range(20):
    for n in range(20):
        print(f"{confusion[t, n]:.2f}", end=" ")
    print()

'test'